In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

matches = list(Path(".").rglob("titanic.csv"))
if not matches:
    raise FileNotFoundError("Extract the practical datasets ZIP beside this notebook.")
df = pd.read_csv(matches[0])
print("Dataset shape:", df.shape)
display(df.head())
print("\nMissing values:\n", df.isnull().sum())

In [ ]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
X, y = df.drop(columns="Survived"), df["Survived"]
numeric = X.select_dtypes(include=np.number).columns.tolist()
categorical = X.select_dtypes(exclude=np.number).columns.tolist()
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)
print("Training shape:", X_train_ready.shape, "| Testing shape:", X_test_ready.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000)
model.fit(X_train_ready, y_train)
pred = model.predict(X_test_ready)
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred))

In [ ]:
print("Practical 1 Summary:")
print("Titanic data was cleaned using median/mode imputation, categorical encoding and feature scaling.")
print("FamilySize, IsAlone and Title were engineered; logistic regression evaluated the processed features.")